# OpenRouter Client — Runbook

**File:** `backend/app/nutrition/openrouter_client.py`  
**Purpose:** Sends diet logs to the OpenRouter API (Claude / Llama) and returns structured nutrition analysis + 7-day meal plan.  
**Used by:** `nutrition/service.py → run_ai_analysis()` (called as a FastAPI BackgroundTask)

---

## How It Works

```
Worker logs diet → POST /nutrition/log
        │
        ▼
  BackgroundTask: run_ai_analysis()
        │
        ▼
  OpenRouterClient.analyze_nutrition()
        │
        ├── Try PRIMARY model  (claude-3-haiku)
        │       │ OK → parse JSON → store in DB
        │       │ Fail ↓
        └── Try FALLBACK model (llama-3-8b)
                │ OK → parse JSON → store in DB
                │ Fail → log error, store ai_analysis_error in DB
```

**Retry logic:** primary model fails → 2-second sleep → fallback model (1 retry total).  
**Timeout:** 30 seconds per HTTP call.  
**Output format:** Structured JSON — deficiencies, 7-day meal plan, referral flag, summary.

---
## 1. Setup

Run this notebook from inside `backend/` with the virtualenv active:
```bash
cd backend
source venv/bin/activate
pip install ipykernel httpx python-dotenv pydantic-settings
jupyter notebook notebooks/openrouter_client_runbook.ipynb
```

In [ ]:
import sys, os

# Make sure `app/` is importable (run from backend/ directory)
sys.path.insert(0, os.path.abspath(".."))

# Load .env so settings picks up OPENROUTER_API_KEY
from dotenv import load_dotenv
load_dotenv("../.env")

from app.config import settings
print("Primary model :", settings.openrouter_primary_model)
print("Fallback model:", settings.openrouter_fallback_model)
print("Base URL      :", settings.openrouter_base_url)
print("API key set   :", bool(settings.openrouter_api_key))

> ⚠️ If **API key set: False**, open `backend/.env` and add `OPENROUTER_API_KEY=sk-or-...`

---
## 2. Connectivity Check

Verify the OpenRouter API is reachable and the key is valid before running any analysis.

In [ ]:
import httpx

async def check_connectivity():
    async with httpx.AsyncClient(timeout=10.0) as client:
        resp = await client.get(
            f"{settings.openrouter_base_url}/models",
            headers={"Authorization": f"Bearer {settings.openrouter_api_key}"},
        )
        if resp.status_code == 200:
            models = resp.json().get("data", [])
            print(f"✅  Connected — {len(models)} models available")
        elif resp.status_code == 401:
            print("❌  401 Unauthorized — check OPENROUTER_API_KEY")
        else:
            print(f"⚠️  Unexpected status: {resp.status_code}")
            print(resp.text[:300])

import asyncio
asyncio.run(check_connectivity())

---
## 3. Normal Analysis — Happy Path

Call `analyze_nutrition()` with a typical SAM-risk child profile and a 7-day diet log.  
Expected: structured JSON with `deficiencies`, `meal_plan`, `referral_needed`, `summary`.

In [ ]:
import json
from app.nutrition.openrouter_client import openrouter_client

SAMPLE_DIET = [
    {"name": "Rice",          "quantity_g": 150},
    {"name": "Dal",           "quantity_g": 60},
    {"name": "Egg",           "quantity_g": 50},
    {"name": "Banana",        "quantity_g": 80},
    {"name": "Ragi porridge", "quantity_g": 100},
]

async def run_happy_path():
    result = await openrouter_client.analyze_nutrition(
        child_name="Kavya",
        age_months=24,
        weight_kg=9.0,
        height_cm=82.0,
        muac_cm=12.0,
        status="MAM",
        diet_log=SAMPLE_DIET,
    )
    return result

result = asyncio.run(run_happy_path())
print(json.dumps(result, indent=2))

In [ ]:
# Validate expected keys
EXPECTED_KEYS = {"deficiencies", "meal_plan", "referral_needed", "summary"}
missing = EXPECTED_KEYS - result.keys()
if missing:
    print(f"❌  Missing keys: {missing}")
else:
    print("✅  All expected keys present")

print(f"   model_used      : {result.get('model_used')}")
print(f"   deficiencies    : {len(result.get('deficiencies', []))} found")
print(f"   meal_plan days  : {len(result.get('meal_plan', []))}")
print(f"   referral_needed : {result.get('referral_needed')}")
print(f"   summary         : {result.get('summary', '')[:80]}...")

---
## 4. Critical Case — SAM + Referral Needed

Test a severely malnourished child with a minimal diet.  
Expected: `referral_needed: true` and `referral_reason` populated.

In [ ]:
SAM_DIET = [
    {"name": "Rice water", "quantity_g": 100},
    {"name": "Salt",       "quantity_g": 2},
]

async def run_sam_case():
    return await openrouter_client.analyze_nutrition(
        child_name="Rajan",
        age_months=18,
        weight_kg=5.8,   # severely underweight for 18 months
        height_cm=72.0,
        muac_cm=10.8,    # below 11.5 cm — SAM threshold
        status="SAM",
        diet_log=SAM_DIET,
    )

sam_result = asyncio.run(run_sam_case())

referral = sam_result.get("referral_needed")
reason   = sam_result.get("referral_reason")
icon = "✅" if referral else "⚠️  (expected True for SAM with MUAC < 11.5)"
print(f"{icon}  referral_needed : {referral}")
print(f"   referral_reason : {reason}")
print(f"   summary         : {sam_result.get('summary', '')}")

---
## 5. Prompt Inspection

Print the exact prompt sent to OpenRouter so you can audit or tune it.

In [ ]:
prompt = openrouter_client._build_prompt(
    child_name="Kavya",
    age_months=24,
    weight_kg=9.0,
    height_cm=82.0,
    muac_cm=12.0,
    status="MAM",
    diet_log=SAMPLE_DIET,
)
print(prompt)

---
## 6. Fallback Model Test

Force the primary model to fail (bad model name) and verify the fallback triggers.

In [ ]:
from app.nutrition.openrouter_client import OpenRouterClient
from app.config import settings

class ForceFallbackClient(OpenRouterClient):
    """Overrides primary model with an invalid name to force fallback."""
    def __init__(self):
        super().__init__()
        self.primary_model = "invalid/model-that-does-not-exist"

fallback_client = ForceFallbackClient()

async def run_fallback_test():
    result = await fallback_client.analyze_nutrition(
        child_name="TestChild",
        age_months=12,
        weight_kg=7.5,
        height_cm=72.0,
        muac_cm=13.0,
        status="Normal",
        diet_log=[{"name": "Milk", "quantity_g": 200}],
    )
    return result

try:
    fb_result = asyncio.run(run_fallback_test())
    print(f"✅  Fallback succeeded — model_used: {fb_result.get('model_used')}")
except Exception as e:
    print(f"❌  Both models failed: {e}")

---
## 7. Error Handling Tests

### 7a. Invalid API key → should raise `RuntimeError`

In [ ]:
class BadKeyClient(OpenRouterClient):
    def __init__(self):
        super().__init__()
        self.api_key = "sk-or-INVALID"
        self.fallback_model = self.primary_model  # same invalid key for both

bad_client = BadKeyClient()

async def run_bad_key():
    return await bad_client.analyze_nutrition(
        child_name="Test", age_months=12, weight_kg=7.0, height_cm=70.0,
        muac_cm=13.0, status="Normal", diet_log=[]
    )

try:
    asyncio.run(run_bad_key())
    print("⚠️  Expected an error but got none")
except RuntimeError as e:
    print(f"✅  Caught RuntimeError (expected): {str(e)[:80]}")
except Exception as e:
    print(f"✅  Caught {type(e).__name__}: {str(e)[:80]}")

### 7b. Malformed JSON response → should raise `ValueError`

In [ ]:
from unittest.mock import AsyncMock, patch, MagicMock

fake_response = MagicMock()
fake_response.status_code = 200
fake_response.json.return_value = {
    "choices": [{"message": {"content": "This is not JSON at all!"}}]
}

async def run_bad_json():
    with patch("httpx.AsyncClient.post", new_callable=AsyncMock, return_value=fake_response):
        return await openrouter_client._call_openrouter("test prompt", "some/model")

try:
    asyncio.run(run_bad_json())
    print("⚠️  Expected ValueError but got none")
except ValueError as e:
    print(f"✅  Caught ValueError (expected): {str(e)[:80]}")

### 7c. Network timeout → should raise `httpx.ReadTimeout`

In [ ]:
import httpx

async def run_timeout():
    async def raise_timeout(*a, **kw):
        raise httpx.ReadTimeout("timed out", request=None)

    with patch("httpx.AsyncClient.post", new_callable=AsyncMock, side_effect=raise_timeout):
        return await openrouter_client._call_openrouter("test", "some/model")

try:
    asyncio.run(run_timeout())
    print("⚠️  Expected timeout but got none")
except httpx.ReadTimeout:
    print("✅  Caught ReadTimeout (expected)")

---
## 8. Latency Benchmark

Measure typical end-to-end call latency. Useful for deciding whether 30s timeout is appropriate.

In [ ]:
import time

async def benchmark(n: int = 3):
    times = []
    for i in range(n):
        t0 = time.perf_counter()
        await openrouter_client.analyze_nutrition(
            child_name="BenchChild",
            age_months=30,
            weight_kg=10.5,
            height_cm=88.0,
            muac_cm=13.5,
            status="Normal",
            diet_log=SAMPLE_DIET,
        )
        elapsed = time.perf_counter() - t0
        times.append(elapsed)
        print(f"  run {i+1}: {elapsed:.2f}s")
    print(f"\n  avg: {sum(times)/len(times):.2f}s | min: {min(times):.2f}s | max: {max(times):.2f}s")

asyncio.run(benchmark(3))

---
## 9. Troubleshooting Guide

| Symptom | Likely Cause | Fix |
|---|---|---|
| `RuntimeError: OpenRouter error 401` | Wrong or missing API key | Set `OPENROUTER_API_KEY` in `backend/.env` |
| `RuntimeError: OpenRouter error 429` | Rate limit hit | Reduce call frequency; upgrade plan; retry logic already handles 1 retry |
| `RuntimeError: OpenRouter error 503` | Model overloaded | Fallback triggers automatically; check OpenRouter status page |
| `ValueError: Invalid JSON from AI` | Model returned plain text instead of JSON | Prompt is strict but models sometimes stray; log `raw_response` and file a model-specific tuning |
| `httpx.ReadTimeout` | API took > 30s | Increase `timeout=30.0` in `_call_openrouter`; check network |
| `ai_analysis` stays `None` in DB | Background task failed silently | Check `ai_analysis_error` field in the `nutrition_logs` document; see server logs |
| Both models fail | Network down or key invalid | Confirm connectivity with Section 2 above |

### Checking for silent AI failures in MongoDB

```python
# Run in a Python shell with DB access
from app.database import get_db
import asyncio

async def find_failed_logs():
    db = get_db()
    failed = await db.nutrition_logs.find(
        {"ai_analysis_error": {"$exists": True}}
    ).to_list(length=50)
    for log in failed:
        print(log["_id"], "|", log["ai_analysis_error"][:80])

asyncio.run(find_failed_logs())
```

---
## 10. Switching Models

Change the active model without redeploying — edit `backend/.env`:

```env
# High accuracy (default)
OPENROUTER_PRIMARY_MODEL=anthropic/claude-3-haiku

# Free-tier fallback
OPENROUTER_FALLBACK_MODEL=meta-llama/llama-3-8b-instruct

# Upgrade options
# OPENROUTER_PRIMARY_MODEL=anthropic/claude-3-sonnet
# OPENROUTER_PRIMARY_MODEL=anthropic/claude-3-opus
# OPENROUTER_PRIMARY_MODEL=google/gemini-pro
```

Restart the FastAPI server after changing `.env`.  
No code changes needed — `config.py` reads these at startup.

---
## 11. Expected JSON Output Schema

The AI must return exactly this shape. Any deviation raises `ValueError` in `_call_openrouter`.

```json
{
  "deficiencies": [
    {
      "nutrient": "Iron",
      "severity": "moderate",
      "foods": ["Moringa leaves", "Sesame seeds", "Horsegram"]
    }
  ],
  "meal_plan": [
    {
      "day": "Monday",
      "breakfast": "Ragi porridge with banana",
      "lunch": "Rice, sambar, moringa stir-fry",
      "snack": "Boiled egg",
      "dinner": "Khichdi with vegetables"
    }
    // ... 6 more days
  ],
  "referral_needed": false,
  "referral_reason": null,
  "summary": "Two-sentence plain-English summary for the Anganwadi worker.",
  "model_used": "anthropic/claude-3-haiku"   // added by client, not the model
}
```

### Severity values
| Value | Meaning |
|---|---|
| `mild` | Early deficiency — dietary change recommended |
| `moderate` | Clear deficiency — dietary change + monitoring |
| `severe` | Clinical deficiency — `referral_needed: true` expected |